# EDA 004.06 — Time Series Exploration

**Understanding temporal structure in sequential data**

## Learning Objectives
By the end of this notebook you will be able to:
- Identify the four components of a time series (trend, seasonality, cyclicity, noise)
- Decompose a time series using additive and multiplicative models
- Test for stationarity using the Augmented Dickey-Fuller (ADF) test
- Compute and interpret rolling statistics
- Read Autocorrelation (ACF) and Partial Autocorrelation (PACF) plots

---

## Time Series Components

| Component | Description | Example |
|---|---|---|
| **Trend** | Long-term increase or decrease | Revenue growing year-over-year |
| **Seasonality** | Fixed, regular patterns | Higher sales every December |
| **Cyclicity** | Irregular multi-year fluctuations | Business cycles |
| **Noise / Residual** | Random unexplained variation | Day-to-day fluctuations |

**Decomposition models:**
- **Additive:** $Y_t = T_t + S_t + R_t$ — use when seasonal amplitude is constant
- **Multiplicative:** $Y_t = T_t \times S_t \times R_t$ — use when seasonal amplitude grows with the trend

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style='whitegrid')
%matplotlib inline

# Seaborn flights dataset — monthly airline passengers 1949-1960
flights = sns.load_dataset('flights')
# Convert to time series with DatetimeIndex
flights['date'] = pd.to_datetime(flights[['year', 'month']].assign(
    month=flights['month'].map({
        'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
        'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12
    }), day=1))
ts = flights.set_index('date')['passengers'].sort_index()

print(f'Period: {ts.index[0].date()} → {ts.index[-1].date()}')
print(f'Observations: {len(ts)}')
ts.head(12)

---
## 1 — Visualise the Time Series

**Reference:** [pandas.Series.plot](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.html)

Always start by plotting the raw series to spot:
- **Trend** — is the series going up or down over time?
- **Seasonality** — does a pattern repeat at fixed intervals?
- **Outliers** — sudden spikes or drops
- **Structural breaks** — sudden change in the level or variance

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Raw time series
ts.plot(ax=axes[0], color='steelblue', lw=1.5)
axes[0].set_title('Monthly Airline Passengers (1949–1960)')
axes[0].set_ylabel('Passengers (thousands)')

# Seasonal subseries — one line per year to highlight seasonal pattern
pivot = flights.pivot(index='month', columns='year', values='passengers')
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot.reindex(month_order)
pivot.plot(ax=axes[1], colormap='tab20', legend=True, lw=1.2)
axes[1].set_title('Seasonal Pattern by Year — each line is one year')
axes[1].set_ylabel('Passengers')
axes[1].legend(title='Year', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

---
## 2 — Seasonal Decomposition

**Reference:** [statsmodels.tsa.seasonal.seasonal_decompose](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)

`seasonal_decompose` splits a series into **trend + seasonal + residual** components.

- `model='additive'` — seasonal amplitude is constant
- `model='multiplicative'` — seasonal amplitude grows proportionally to the level

The airline passenger data has growing seasonality → **multiplicative** model is appropriate.

In [ ]:
decomp = seasonal_decompose(ts, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)

decomp.observed.plot(ax=axes[0], color='steelblue', lw=1.5)
axes[0].set_title('Observed')

decomp.trend.plot(ax=axes[1], color='darkorange', lw=2)
axes[1].set_title('Trend')

decomp.seasonal.plot(ax=axes[2], color='seagreen', lw=1.5)
axes[2].set_title('Seasonal (multiplicative factor)')

decomp.resid.plot(ax=axes[3], color='coral', lw=1)
axes[3].set_title('Residual (noise)')
axes[3].axhline(1, color='black', linestyle='--', lw=0.8)

plt.suptitle('Multiplicative Decomposition — Airline Passengers',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Peak seasonal month
seasonal_factors = decomp.seasonal[:12].set_axis(
    pd.date_range('2000-01', periods=12, freq='MS'))
print('Seasonal factors (1.0 = average):')
for month, val in seasonal_factors.items():
    print(f'  {month.strftime("%b")}: {val:.3f}')

---
## 3 — Stationarity and the ADF Test

**Reference:** [statsmodels.tsa.stattools.adfuller](https://www.statsmodels.org/stable/generated/statsmodels.tsa.stattools.adfuller.html)

A **stationary** time series has constant mean and variance over time — a required assumption for many forecasting models (ARIMA, etc.).

The **Augmented Dickey-Fuller (ADF) test** tests:
- $H_0$: The series has a unit root (non-stationary)
- $H_1$: The series is stationary

If $p$-value < 0.05 → reject $H_0$ → series is **stationary**.

Common fixes for non-stationarity:
- **Differencing** — compute $y_t - y_{t-1}$
- **Log transform** → then difference
- **Seasonal differencing** — compute $y_t - y_{t-12}$

In [ ]:
def adf_test(series, name='Series'):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'{name}:')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical values: 1%={result[4]["1%"]:.3f}, '
                              f'5%={result[4]["5%"]:.3f}, '
                              f'10%={result[4]["10%"]:.3f}')
    print(f'  Result        : {"STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"}')
    print()

# Raw series — non-stationary
adf_test(ts, 'Raw passengers')

# Log differenced — stationary
ts_log = np.log(ts)
ts_log_diff = ts_log.diff(1)
adf_test(ts_log_diff, 'Log + first-difference')

# Plot transformation
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ts.plot(ax=axes[0], color='steelblue', title='Raw (non-stationary — trend + seasonality)')
ts_log.plot(ax=axes[1], color='darkorange', title='Log transform (stabilises variance)')
ts_log_diff.plot(ax=axes[2], color='seagreen', title='Log + First Difference (stationary)')
axes[2].axhline(0, color='black', linestyle='--', lw=0.8)

plt.tight_layout()
plt.show()

---
## 4 — Rolling Statistics

**Reference:** [pandas.Series.rolling](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html)

Rolling statistics compute statistics over a **sliding window** of fixed width:

| Statistic | Use |
|---|---|
| Rolling mean | Smooth noise, reveal trend |
| Rolling std | Measure local volatility |
| Rolling min/max | Detect extremes in window |

> Checking if rolling mean and std are stable over time is a **visual stationarity check**.

In [ ]:
window = 12  # 12-month rolling window

rolling_mean = ts.rolling(window=window).mean()
rolling_std  = ts.rolling(window=window).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ts.plot(ax=axes[0], color='steelblue', alpha=0.5, label='Original')
rolling_mean.plot(ax=axes[0], color='red', lw=2, label=f'{window}-month Rolling Mean')
axes[0].fill_between(ts.index,
                      rolling_mean - rolling_std,
                      rolling_mean + rolling_std,
                      alpha=0.15, color='red', label='±1 Std Dev')
axes[0].set_title('Rolling Mean ± Std Dev')
axes[0].legend()

rolling_std.plot(ax=axes[1], color='darkorange', lw=1.5)
axes[1].set_title('Rolling Std — growing = heteroscedastic (non-stationary variance)')

plt.tight_layout()
plt.show()

---
## 5 — ACF and PACF Plots

**Reference:** [statsmodels — ACF/PACF](https://www.statsmodels.org/stable/graphics.html#autocorrelation-plots)

**Autocorrelation (ACF)** measures the correlation between a series and its own lagged versions:
- Reveals the overall memory structure — how long past values influence the present

**Partial Autocorrelation (PACF)** shows the direct effect of each lag, removing the indirect effects of intermediate lags:
- Used to identify the **AR order** (p) in ARIMA models

| Model | ACF signature | PACF signature |
|---|---|---|
| AR(p) | Tail off gradually | Cut off at lag p |
| MA(q) | Cut off at lag q | Tail off gradually |
| ARMA(p,q) | Tail off | Tail off |

In [ ]:
# Use the stationary (log-differenced) series for ACF/PACF
ts_stationary = ts_log_diff.dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

plot_acf(ts_stationary, lags=36, alpha=0.05, ax=axes[0],
         color='steelblue', vlines_kwargs={'colors': 'steelblue'})
axes[0].set_title('ACF — Log-differenced Airline Passengers\n(blue band = 95% CI; spikes outside band are significant)')

plot_pacf(ts_stationary, lags=36, alpha=0.05, ax=axes[1], method='ywm',
          color='coral', vlines_kwargs={'colors': 'coral'})
axes[1].set_title('PACF — Log-differenced Airline Passengers')

plt.tight_layout()
plt.show()

print('Observation: Significant spikes at lags 12, 24 in ACF → strong annual seasonality')

---
## Key Takeaways

- Always **plot** the raw series first — look for trend, seasonality, outliers, and structural breaks
- Use **seasonal subseries plots** to see whether the seasonal pattern is consistent
- **Multiplicative decomposition** when seasonal amplitude grows; additive when it's constant
- Most forecasting models (ARIMA) require **stationarity** — test with ADF, transform if needed
- **Rolling mean/std** provide a visual stationarity check
- **ACF** shows overall autocorrelation structure; **PACF** helps identify ARIMA order

---
## Further Reading

- [Kaggle — Time Series Course](https://www.kaggle.com/learn/time-series)
- [statsmodels Time Series documentation](https://www.statsmodels.org/stable/tsa.html)
- [Forecasting: Principles and Practice (free online)](https://otexts.com/fpp3/) — Hyndman & Athanasopoulos
- [Introduction to Time Series and Forecasting](https://link.springer.com/book/10.1007/978-3-319-29854-2) — Brockwell & Davis
- [StatQuest: ACF/PACF explained](https://www.youtube.com/results?search_query=statquest+acf+pacf)